# Comparing `pgmuvi`, `celerite2`, and `tinygp` on synthetic light curves

This tutorial responds to the referee request for a more quantitative and practical comparison.
We compare usage on simple synthetic datasets and show where each tool is most appropriate.


## Mathematical assumptions and scope

All three tools model data with Gaussian processes (GPs):

$$y(\mathbf{x}) \sim \mathcal{GP}(m(\mathbf{x}), k(\mathbf{x}, \mathbf{x}')),$$

with a mean function $m$ and covariance kernel $k$.

- **`pgmuvi`** wraps GPyTorch models with astronomy-oriented defaults
  (spectral-mixture initialisation, 1D/2D model shortcuts, and convenience methods).
- **`celerite2`** is extremely efficient for 1D time-series kernels in its supported
  semiseparable form, but it is not a general 2D multiwavelength GP framework.
- **`tinygp`** is flexible and can model multi-input problems, but users must define
  more model structure manually.


In [ ]:
import textwrap
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from pgmuvi.synthetic import make_chromatic_sinusoid_2d, make_simple_sinusoid_1d

try:
    from celerite2 import GaussianProcess as CeleriteGaussianProcess
    from celerite2 import terms as celerite_terms
    HAVE_CELERITE = True
except Exception:
    HAVE_CELERITE = False

try:
    import jax.numpy as jnp
    from tinygp import GaussianProcess as TinyGaussianProcess
    from tinygp import kernels as tiny_kernels
    HAVE_TINYGP = True
except Exception:
    HAVE_TINYGP = False

print(f"celerite2 available: {HAVE_CELERITE}")
print(f"tinygp available:   {HAVE_TINYGP}")

def count_code_lines(snippet: str) -> int:
    lines = [ln for ln in textwrap.dedent(snippet).splitlines() if ln.strip()]
    return len(lines)


In [ ]:
# Synthetic single-band (1D) dataset
lc_1d = make_simple_sinusoid_1d(
    n_obs=120,
    period=40.0,
    amplitude=1.0,
    noise_level=0.2,
    noise_type="gaussian",
    irregular=True,
    seed=1,
)

# Synthetic multiwavelength (2D: time + wavelength) dataset
lc_2d = make_chromatic_sinusoid_2d(
    n_per_band=[80, 70, 60],
    period=40.0,
    wavelengths=[0.5, 0.8, 1.2],
    amplitude_law="linear",
    amplitude=1.0,
    amplitude_slope=0.6,
    noise_level=0.15,
    t_span=180.0,
    irregular=True,
    seed=2,
)

print("1D points:", len(lc_1d.xdata))
print("2D points:", len(lc_2d.xdata), "with ndim=", lc_2d.ndim)


## 1D (single-wavelength) comparison

All three tools can handle this case. We compare short, minimal workflows and
record simple wall-clock timings for one pass.


In [ ]:
timings = {}

# --- pgmuvi 1D ---
t0 = time.perf_counter()
res_1d_pgmuvi = lc_1d.fit(model="1D", num_mixtures=2, training_iter=80, miniter=30)
timings["pgmuvi_1d_fit_s"] = time.perf_counter() - t0

print("pgmuvi final loss:", float(res_1d_pgmuvi["loss"][-1]))
print("pgmuvi 1D fit time (s):", round(timings["pgmuvi_1d_fit_s"], 3))


In [ ]:
# --- celerite2 1D (if installed) ---
if HAVE_CELERITE:
    x = np.asarray(lc_1d.xdata.detach().cpu())
    y = np.asarray(lc_1d.ydata.detach().cpu())
    yerr = np.asarray(lc_1d.yerr.detach().cpu())

    kernel = celerite_terms.SHOTerm(sigma=np.std(y), rho=20.0, tau=20.0)
    gp_celerite = CeleriteGaussianProcess(kernel, mean=float(np.mean(y)))

    t0 = time.perf_counter()
    gp_celerite.compute(x, yerr=yerr)
    ll = gp_celerite.log_likelihood(y)
    timings["celerite2_1d_compute_s"] = time.perf_counter() - t0

    print("celerite2 log-likelihood:", float(ll))
    print("celerite2 compute time (s):", round(timings["celerite2_1d_compute_s"], 3))
else:
    print("celerite2 not installed in this environment.")


In [ ]:
# --- tinygp 1D (if installed) ---
if HAVE_TINYGP:
    x = np.asarray(lc_1d.xdata.detach().cpu())
    y = np.asarray(lc_1d.ydata.detach().cpu())
    yerr = np.asarray(lc_1d.yerr.detach().cpu())

    kernel = tiny_kernels.ExpSquared(scale=15.0)

    t0 = time.perf_counter()
    gp_tiny = TinyGaussianProcess(kernel, jnp.asarray(x), diag=jnp.asarray(yerr**2))
    cond = gp_tiny.condition(jnp.asarray(y), jnp.asarray(x))
    pred_mean = np.asarray(cond.gp.loc)
    timings["tinygp_1d_condition_s"] = time.perf_counter() - t0

    print("tinygp predictive mean shape:", pred_mean.shape)
    print("tinygp condition time (s):", round(timings["tinygp_1d_condition_s"], 3))
else:
    print("tinygp not installed in this environment.")


## 2D (multiwavelength) comparison

Here the differences are clearer:

- **`pgmuvi`** has a dedicated 2D model shortcut (`model="2D"`).
- **`celerite2`** is designed for 1D semiseparable kernels and does not directly take
  `(time, wavelength)` coordinates.
- **`tinygp`** can do 2D, but the user must manually choose and configure the
  multi-input kernel and GP object.


In [ ]:
# --- pgmuvi 2D ---
t0 = time.perf_counter()
res_2d_pgmuvi = lc_2d.fit(model="2D", num_mixtures=3, training_iter=120, miniter=40)
timings["pgmuvi_2d_fit_s"] = time.perf_counter() - t0

print("pgmuvi 2D final loss:", float(res_2d_pgmuvi["loss"][-1]))
print("pgmuvi 2D fit time (s):", round(timings["pgmuvi_2d_fit_s"], 3))


In [ ]:
# --- celerite2 limitation for multiwavelength inputs ---
if HAVE_CELERITE:
    x2 = np.asarray(lc_2d.xdata.detach().cpu())
    y2 = np.asarray(lc_2d.ydata.detach().cpu())
    yerr2 = np.asarray(lc_2d.yerr.detach().cpu())
    kernel = celerite_terms.SHOTerm(sigma=np.std(y2), rho=20.0, tau=20.0)
    gp_celerite_2d = CeleriteGaussianProcess(kernel, mean=float(np.mean(y2)))

    try:
        gp_celerite_2d.compute(x2, yerr=yerr2)
        print("Unexpected success: celerite2 accepted 2D coordinates.")
    except Exception as exc:
        print("Expected celerite2 limitation for 2D inputs:")
        print(type(exc).__name__, str(exc)[:180])
else:
    print("celerite2 not installed in this environment.")


In [ ]:
# --- tinygp 2D from scratch (if installed): explicit kernel design ---
if HAVE_TINYGP:
    x2 = np.asarray(lc_2d.xdata.detach().cpu())
    y2 = np.asarray(lc_2d.ydata.detach().cpu())
    yerr2 = np.asarray(lc_2d.yerr.detach().cpu())

    # User has to define a multi-input kernel manually.
    kernel_2d = tiny_kernels.ExpSquared(scale=0.6)

    t0 = time.perf_counter()
    gp_tiny_2d = TinyGaussianProcess(
        kernel_2d,
        jnp.asarray(x2),
        diag=jnp.asarray(yerr2**2),
        mean=float(np.mean(y2)),
    )
    cond_2d = gp_tiny_2d.condition(jnp.asarray(y2), jnp.asarray(x2))
    _ = np.asarray(cond_2d.gp.loc)
    timings["tinygp_2d_condition_s"] = time.perf_counter() - t0

    print("tinygp 2D condition time (s):", round(timings["tinygp_2d_condition_s"], 3))
else:
    print("tinygp not installed in this environment.")


In [ ]:
pgmuvi_1d_snippet = """
res_1d_pgmuvi = lc_1d.fit(
    model="1D",
    num_mixtures=2,
    training_iter=80,
    miniter=30,
)
"""

pgmuvi_2d_snippet = """
res_2d_pgmuvi = lc_2d.fit(
    model="2D",
    num_mixtures=3,
    training_iter=120,
    miniter=40,
)
"""

tinygp_2d_snippet = """
kernel_2d = tiny_kernels.ExpSquared(scale=0.6)
gp_tiny_2d = TinyGaussianProcess(
    kernel_2d,
    jnp.asarray(x2),
    diag=jnp.asarray(yerr2**2),
    mean=float(np.mean(y2)),
)
cond_2d = gp_tiny_2d.condition(jnp.asarray(y2), jnp.asarray(x2))
"""

print("Representative non-empty code lines")
print("pgmuvi 1D:", count_code_lines(pgmuvi_1d_snippet))
print("pgmuvi 2D:", count_code_lines(pgmuvi_2d_snippet))
print("tinygp 2D:", count_code_lines(tinygp_2d_snippet))

print("\nTiming summary (if package available):")
for key, val in timings.items():
    print(f"{key:>24s}: {val:.3f} s")


## Practical takeaway

- Prefer **`pgmuvi`** when you want a fast astronomy workflow with built-in 1D/2D
  modelling defaults, especially for multiwavelength light curves.
- Prefer **`celerite2`** for very fast 1D analyses in its kernel class.
- Prefer **`tinygp`** when you need total control and are willing to write more
  model-definition code.
